# CANopen — SDO, PDO, NMT and heartbeats

A master and a simulated CiA 401-style I/O module share a virtual CAN bus. Replace the bus with `await CanBus.OpenAsync("socketcan:can0")` (or `slcan:`, `gsusb:`, `pcan:usb1`) to reach real nodes.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.19.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.19.0-preview.1"

## Boot a node and scan the network
The module sends a boot-up message, then heartbeats; the scan reads 0x1000, 0x1008 and 0x1018 over SDO.

In [ ]:
using IoTCom.Net.Protocols.CanOpen;
using IoTCom.Net.Transport.Can;

var coNet = new VirtualCanNetwork("notebook");
var ioModule = CanOpenIoModuleSimulator.Create(coNet.CreateNode(), nodeId: 5, heartbeatMs: 200, eventTimerMs: 150);
var coBus = coNet.CreateNode();
var coMaster = CanOpenMaster.Create(coBus);
await coMaster.StartAsync();
await ioModule.StartAsync();
foreach (var n in await coMaster.ScanAsync(1, 10, TimeSpan.FromMilliseconds(50)))
    Console.WriteLine($"node {n.Id}: {n.Name}, type 0x{n.DeviceType:X8}, vendor 0x{n.Identity?.Vendor:X8}, serial {n.Identity?.Serial:X8}, {n.State}");

## SDO: expedited, segmented and aborts
Values up to 4 bytes fit one frame; longer ones are split into 7-byte segments with a toggle bit.

In [ ]:
Console.WriteLine(await coMaster.ReadAsync(5, 0x6401, 1, CanOpenDataType.Integer16));             // expedited
Console.WriteLine(await coMaster.ReadAsync(5, 0x2100, 0, CanOpenDataType.VisibleString));          // segmented
await coMaster.WriteAsync(5, 0x2100, 0, CanOpenDataType.VisibleString, "Pump skid 9, Bekasi");
Console.WriteLine(ioModule.Node.Dictionary[0x2100, 0]);
try { await coMaster.UploadAsync(5, 0x9999, 0); } catch (CanOpenSdoException e) { Console.WriteLine(e.Message); }
try { await coMaster.WriteAsync(5, 0x1000, 0, CanOpenDataType.Unsigned32, 1); } catch (CanOpenSdoException e) { Console.WriteLine(e.Message); }

## NMT start, PDOs and SYNC
Only operational nodes send PDOs. TPDO1 is event-driven with an event timer; TPDO2 answers every SYNC.

In [ ]:
var mapping = await coMaster.ReadTpdoMappingAsync(5, 1);
Console.WriteLine(string.Join(", ", mapping.Objects));
var pdoCount = 0;
coMaster.PdoReceived += (node, pdo, data) => { if (pdoCount++ < 4) Console.WriteLine($"TPDO{pdo} node {node}: {Convert.ToHexString(data)}"); };
await coMaster.NmtAsync(NmtCommand.Start, 5);
await coMaster.WriteAsync(5, 0x6200, 1, CanOpenDataType.Unsigned8, 1);                             // pump on
ioModule.Step();
await coMaster.SyncAsync();
await Task.Delay(400);
Console.WriteLine($"flow {ioModule.Node.Dictionary[0x6401, 2].Value} (0.1 l/min), state {coMaster.Nodes.First().State}");

## On the wire

In [ ]:
foreach (var f in CanOpenCodec.Describe(new CanFrame(0x585, Convert.FromHexString("410810000B000000"))))
    Console.WriteLine($"{f.Name,-8} {f.Value}");
await ioModule.DisposeAsync();
await coMaster.DisposeAsync();

## Going further
`iotcom canopen scan --can sim`, `iotcom canopen monitor --can sim`, the Gallery's *CANopen I/O modules* and the CanOpenMaster sample. See `docs/en/protocols/canopen.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*